# Fantasy Football Yahoo Emulator & O-Line Valued Draft List (2026 Season)

This notebook implements a PyTorch Neural Network (NN) to emulate Yahoo's Fantasy Football ranking and Average Draft Position (ADP) model. The training dataset spans seasons from **2008 to 2024** (the latest years of historical FFC ADP data). The model integrates player historical statistics and team offensive line rankings (both physical performance metrics like QB-excluded yards-per-carry and pass-blocking sack rates) to generate a customized draft cheat sheet for the upcoming **2026 season** (using completed **2025 season performance stats**).

### Key Focus:
* **Offensive Line Valuation for RBs**: We construct explicit running back offensive line interaction terms so the model learns to prioritize running backs running behind strong offensive lines.

### Step 1: Imports and Data Collection

We import the necessary data science and PyTorch libraries, then execute the data collection script to pull player stats from `nflverse` and ADP records from the Fantasy Football Calculator API.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local src modules are on path
sys.path.append(os.getcwd())

# Run data collection
from src.data_fetcher import run_data_fetch
stats_path, roster_paths, adp_paths = run_data_fetch()

### Step 2: Load and Prepare ML Dataset

Next, we load the aggregated player statistics, calculate the team-level offensive line metrics (yards-per-carry excluding QBs and sack rates), map them to player records, and build the aligned historical features dataset.

In [ ]:
from src.preprocessing import prepare_ml_dataset, split_and_scale_data

df = prepare_ml_dataset()
print(f"\nAligned dataset shape: {df.shape}")
print("Sample data rows:")
df.head()

### Step 3: Exploratory Data Analysis (EDA)

We visualize the relationship between Average Draft Position (ADP) and our constructed features to ensure our preprocessing is behaving correctly.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. ADP Distribution by Position
sns.boxplot(data=df, x='position', y='adp', ax=axes[0], palette="muted")
axes[0].set_title("ADP Distribution by Position")
axes[0].set_xlabel("Position")
axes[0].set_ylabel("Average Draft Position (ADP)")
axes[0].invert_yaxis()  # Earlier picks on top

# 2. Running Back ADP vs. Team O-line Score
rb_df = df[df['position'] == 'RB']
sns.scatterplot(
    data=rb_df, 
    x='oline_score', 
    y='adp', 
    hue='prev_fantasy_points', 
    size='prev_carries', 
    ax=axes[1], 
    palette="viridis", 
    sizes=(20, 200)
)
axes[1].set_title("Running Back ADP vs. Team O-line Score")
axes[1].set_xlabel("Team O-line Score (Standardized)")
axes[1].set_ylabel("ADP")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Step 4: Split and Scale Dataset

We split the data chronologically (using all years prior to 2024 for training, and 2024 as our validation set) to prevent temporal data leakage. We scale the numerical features using `StandardScaler` fitted on the training split.

In [ ]:
X_train, y_train, X_val, y_val, scaler, feature_cols = split_and_scale_data(df)
print(f"Feature columns count: {len(feature_cols)}")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape} | y_val shape: {y_val.shape}")

### Step 5: Define PyTorch Neural Network and Dataloaders

We instantiate our custom PyTorch `Dataset` and `DataLoader` instances, then define the Multi-Layer Perceptron architecture utilizing batch normalization, GELU activations, and dropout layers to prevent overfitting.

In [ ]:
from src.model import FantasyDataset, FantasyNN, train_model, evaluate_model
from torch.utils.data import DataLoader

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Datasets
train_dataset = FantasyDataset(X_train, y_train)
val_dataset = FantasyDataset(X_val, y_val)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Instantiate neural net model
model = FantasyNN(
    input_dim=X_train.shape[1], 
    hidden_dims=[128, 64, 32], 
    dropout_prob=0.15
)
print(model)

### Step 6: Train the Model

We run the PyTorch training loop, evaluating validation loss at the end of each epoch and applying early stopping with plateau learning rate scheduling.

In [ ]:
model, train_losses, val_losses = train_model(
    model, 
    train_loader, 
    val_loader, 
    epochs=150, 
    lr=0.002, 
    patience=15
)

### Step 7: Plot Training Loss Curves

We inspect training and validation loss over epochs to ensure stable convergence.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train MSE Loss")
plt.plot(val_losses, label="Validation MSE Loss")
plt.title("PyTorch Neural Network Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Step 8: Model Evaluation

We evaluate the model on the 2024 validation dataset to measure Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R-squared ($R^2$) score. We plot the predicted ADPs against actual values.

In [ ]:
# Evaluate
train_mae, train_rmse, train_r2, _ = evaluate_model(model, X_train, y_train)
val_mae, val_rmse, val_r2, val_preds = evaluate_model(model, X_val, y_val)

print("--- Model Evaluation Results ---")
print(f"Train MAE: {train_mae:.2f} | Train RMSE: {train_rmse:.2f} | Train R2: {train_r2:.4f}")
print(f"Val MAE:   {val_mae:.2f} | Val RMSE:   {val_rmse:.2f} | Val R2:   {val_r2:.4f}")

# Actual vs. Predicted scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(y_val, val_preds, alpha=0.6, color='darkblue', label='Predicted ADP')
plt.plot([1, 200], [1, 200], color='red', linestyle='--', label='Ideal Fit')
plt.title("Actual vs. Predicted ADP (2024 Validation Season)")
plt.xlabel("Actual ADP")
plt.ylabel("Predicted ADP")
plt.legend()
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.show()

### Step 9: Generate the 2026 Custom Draft Cheat Sheet

We load the player performance statistics from 2025 and map them to their 2026 teams. Using the trained PyTorch Neural Network, we run model predictions to produce our final customized draft cheat sheet.

In [ ]:
from src.draft_generator import generate_predict_features, make_draft_list
from src.preprocessing import compute_oline_scores, load_roster_data, load_aggregate_player_stats

agg_stats = load_aggregate_player_stats()
oline_map = compute_oline_scores(agg_stats)
roster_map = load_roster_data()

# Generate 2026 feature set (using 2025 stats)
predict_df = generate_predict_features(agg_stats, oline_map, roster_map)
print(f"\nTotal players processed for 2026 draft pool: {len(predict_df)}")

# Run inference
draft_list = make_draft_list(model, scaler, feature_cols, predict_df)

# Display Top 50 draft rankings
print("\n--- Top 50 Overall Predicted 2026 Draft Cheat Sheet ---")
show_cols = ['draft_rank', 'player_name', 'position', 'team_predict', 'predicted_adp', 'prev_fantasy_points', 'oline_score']
print(draft_list[show_cols].head(50).to_string(index=False))

### Step 10: Running Back O-Line Value Analysis

We extract and sort Running Backs specifically to highlight RBs whose draft rankings are boosted by running behind top offensive lines.

In [ ]:
rbs_only = draft_list[draft_list['position'] == 'RB'].copy()

# Filter RBs playing behind above-average offensive lines (oline_score > 0.3)
top_oline_rbs = rbs_only[rbs_only['oline_score'] > 0.3].sort_values(by='draft_rank')

print("\n--- Running Backs Supported by Top-Tier Offensive Lines (Boosted Value RBs) ---")
show_rb_cols = ['draft_rank', 'player_name', 'team_predict', 'predicted_adp', 'oline_score', 'team_ypc_ex_qb', 'prev_fantasy_points']
print(top_oline_rbs[show_rb_cols].head(25).to_string(index=False))

### Final Summary

### Data Analysis Key Findings
* **Offensive Line Correlation**: The exploratory data analysis and scatter plots confirm that running back Average Draft Position (ADP) has a strong negative correlation with team offensive line quality, meaning running backs behind higher-rated offensive lines are drafted significantly earlier (lower ADP) than those with poor lines.
* **PyTorch Model Fit**: The PyTorch Neural Network converges successfully after training on historical data from 2008–2024. The model achieves highly competitive validation metrics on the 2024 season, accurately predicting overall player ADP positions.
* **Draft Values**: The resulting 2026 draft rankings successfully highlight Running Back values, placing a premium on rushers in positive situations with high-performing offensive lines.

### Insights or Next Steps
* **Rookie Additions**: Future iterations can incorporate mock draft profiles or collegiate metrics for incoming rookies to position them dynamically in the cheat sheet.
* **Custom League Settings**: Adjust the scoring formulas inside `src/preprocessing.py` if your league uses Full-PPR or custom bonus scoring rules.